In [0]:
%sql
-- Somethings to note. It appears we have some duplicating items in our dataset. We see that some prd keys appear twice. The first two rows do not have a price or product end date and do not appear anywhere in the sales details table.
SELECT 
    * 
FROM workspace.`bike-bronze`.crm_prd_info

In [0]:
%sql
-- It appears that where ever we see prd_end_dt as null we can assume that the product is still active.
-- For all others, start and end date appear to be swapped columns.
SELECT 
    * 
FROM workspace.`bike-bronze`.crm_prd_info 
WHERE prd_end_dt IS NULL

In [0]:
%sql
-- Right off the bat we can see that we have 397 records and a corresponding 397 unique product IDS.
-- However, it looks like product keys are not unique. We see product keys duplicated becaues it appears that once we have price raises we have a new product key.
-- This is not ideal as we will have to join on product ID to get the correct price for a given product.
-- We will have to do some more work to clean up our data.
SELECT
    COUNT(*) AS total_records,
    COUNT(prd_id) AS total_product_ids,
    COUNT(DISTINCT prd_id) AS unique_product_ids,
    COUNT(prd_key) AS total_product_keys,
    COUNT(DISTINCT prd_key) AS unique_product_keys
FROM workspace.`bike-bronze`.crm_prd_info;

In [0]:
%sql
-- Product key in the sales details table does not match the product info table. Product ID does not appear either. We will have to modify product key for the product info table to match the sales details table.
SELECT
    *
FROM workspace.`bike-bronze`.crm_sales_details

In [0]:
%sql
-- We have 77 keys that have duplicates.
SELECT
    prd_key,
    COUNT(prd_key) AS key_cnt
FROM workspace.`bike-bronze`.crm_prd_info
GROUP BY prd_key
HAVING
    COUNT(prd_key) > 1

In [0]:
%sql
SELECT 
    * 
FROM workspace.`bike-bronze`.crm_prd_info 
WHERE prd_end_dt IS NOT NULL

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

df = spark.sql("SELECT * FROM workspace.`bike-bronze`.crm_prd_info")
df.display()

In [0]:
# Trimming out any string fields
for field in df.schema.fields:
    if (field.dataType, StringType):
        df = df.withColumn(field.name, trim(field.name))
df.display()

In [0]:
df = df.withColumn("cat_id",F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-","_"))
df = df.withColumn("prd_key",F.substring(col("prd_key"), 7, F.length(col("prd_key"))))
df.display()

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))
df.display()                   

In [0]:
%sql
SELECT
    DISTINCT
        PRD_LINE
FROM workspace.`bike-bronze`.crm_prd_info
    
-- We can see that the start and end dates are swapped.

In [0]:
df = df.withColumn("prd_line", 
                   F.when(F.upper(col("prd_line")) == "R", "Road")
                   .when(F.upper(col("prd_line")) == "T", "Touring")
                   .when(F.upper(col("prd_line")) == "M", "Mountain")
                   .when(F.upper(col("prd_line")) == "S", "Other Sales")
                   .otherwise("n/a"))
df.display()

In [0]:
RENAME_MAP = {
    "prd_id" : "product_id",
    "prd_key" : "product_key",
    "prd_nm" : "product_name",
    "prd_cost" : "product_cost",
    "prd_line" : "product_line",
    "prd_start_dt" : "product_start_date",
    "prd_end_dt" : "product_end_date",
    "cat_id" : "category_id"
}

for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.limit(10).display()

In [0]:
df = df.withColumn("product_start_date", F.to_date(col("product_start_date")))
df = df.withColumn("product_end_date", F.to_date(col("product_end_date")))

df.display()